# 🔥 ForgeFactory — Classifier Agent LoRA Fine-Tuning
**Model**: Mistral-7B-v0.3  
**Method**: LoRA (fp16, no quantization needed on T4 with rank=8)  
**Task**: Fine-tune classifier agent to ask smart clarifying questions + output structured JSON  

> Upload `classifier_training_data_v2.json` to Colab before running

## Step 1 — Install Dependencies

In [ ]:
!pip install -q transformers==4.40.0 peft==0.10.0 trl==0.8.6 datasets==2.19.0 accelerate==0.29.3 bitsandbytes==0.43.1 sentencepiece protobuf

## Step 2 — Check GPU

In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {total:.1f} GB")
else:
    print("⚠️  No GPU detected. Go to Runtime > Change runtime type > T4 GPU")

## Step 3 — Load & Format Dataset

In [ ]:
import json
from datasets import Dataset

# ── Load JSON ──────────────────────────────────────────────────────────
DATA_PATH = "classifier_training_data_v2.json"  # upload this file to Colab

with open(DATA_PATH, "r") as f:
    raw_data = json.load(f)

print(f"Loaded {len(raw_data)} samples")

# ── Format into Mistral instruction template ───────────────────────────
# Mistral expects: <s>[INST] {instruction} [/INST] {output}</s>
def format_sample(sample):
    instruction = sample["instruction"]
    output      = sample["output"]
    # Include 'input' field if non-empty
    if sample.get("input", "").strip():
        user_msg = f"{instruction}\n\nContext: {sample['input']}"
    else:
        user_msg = instruction

    text = f"<s>[INST] {user_msg} [/INST] {output}</s>"
    return {"text": text}

formatted = [format_sample(s) for s in raw_data]
dataset   = Dataset.from_list(formatted)

# ── Train / Val split (90/10) ──────────────────────────────────────────
split   = dataset.train_test_split(test_size=0.1, seed=42)
train_ds = split["train"]
val_ds   = split["test"]

print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")
print("\nSample formatted text:")
print(train_ds[0]["text"][:500])

## Step 4 — Load Model & Tokenizer (4-bit QLoRA for T4)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

MODEL_ID = "mistralai/Mistral-7B-v0.3"

# ── QLoRA config (4-bit) — fits on 15GB T4 ────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# ── Load tokenizer ─────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token     = tokenizer.eos_token   # Mistral has no pad token
tokenizer.padding_side  = "right"               # pad right for causal LM

# ── Load model in 4-bit ────────────────────────────────────────────────
print("Loading Mistral-7B in 4-bit... (~5 min on first run)")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
print("✅ Model loaded")

## Step 5 — Attach LoRA Adapter

In [ ]:
# ── LoRA config ────────────────────────────────────────────────────────
# r=16 gives good capacity for task-specific behaviour
# target_modules: Mistral attention projections
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,           # scaling = alpha/r = 2x
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",  # MLP layers too
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)

# Print trainable parameter count
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## Step 6 — Train

In [ ]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "./classifier_adapter"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    # ── Core training ─────────────────────────────
    num_train_epochs=5,              # more epochs since dataset is small
    per_device_train_batch_size=2,   # T4 safe with 4-bit
    gradient_accumulation_steps=8,   # effective batch = 16
    per_device_eval_batch_size=2,

    # ── Optimizer ─────────────────────────────────
    learning_rate=2e-4,              # standard for LoRA fine-tuning
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    optim="paged_adamw_8bit",        # memory-efficient optimizer
    weight_decay=0.01,

    # ── Sequence length ───────────────────────────
    max_seq_length=1024,             # enough for instruction + output
    dataset_text_field="text",
    packing=False,                   # don't pack — we want clean samples

    # ── Precision ─────────────────────────────────
    fp16=True,
    bf16=False,                      # T4 doesn't support bf16

    # ── Logging & saving ──────────────────────────
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",                # disable wandb
    dataloader_pin_memory=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
)

print("🚀 Starting training...")
trainer.train()
print("✅ Training complete!")

## Step 7 — Save Adapter Only

In [ ]:
# Save ONLY the LoRA adapter weights (~50-200MB, not the full 14GB model)
ADAPTER_SAVE_PATH = "./classifier_adapter_final"

model.save_pretrained(ADAPTER_SAVE_PATH)
tokenizer.save_pretrained(ADAPTER_SAVE_PATH)

print(f"✅ Adapter saved to {ADAPTER_SAVE_PATH}")

# Show saved files and size
import os
total_size = 0
for f in os.listdir(ADAPTER_SAVE_PATH):
    size = os.path.getsize(os.path.join(ADAPTER_SAVE_PATH, f))
    total_size += size
    print(f"  {f}: {size/1e6:.1f} MB")
print(f"Total adapter size: {total_size/1e6:.1f} MB")

## Step 8 — Download Adapter to Local Machine

In [ ]:
import shutil

# Zip the adapter folder for easy download
shutil.make_archive("classifier_adapter_final", "zip", ".", "classifier_adapter_final")
print("✅ Zipped adapter")

# Download
from google.colab import files
files.download("classifier_adapter_final.zip")
print("📦 Download started")

## Step 9 — Test the Fine-Tuned Adapter

In [ ]:
from transformers import pipeline
import torch

def run_classifier_agent(user_instruction: str, max_new_tokens: int = 300) -> str:
    """
    Run the fine-tuned classifier agent on a new instruction.
    Returns clarifying questions + structured JSON.
    """
    prompt = f"<s>[INST] {user_instruction} [/INST]"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,         # low temp for structured output
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the new tokens (not the prompt)
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


# ── Test cases ────────────────────────────────────────────────────────
test_instructions = [
    "Build a system to detect fake reviews on Amazon",
    "I want to predict patient survival rates",
    "Make me an AI for my restaurant",
    "Classify satellite images of forests",
]

for instruction in test_instructions:
    print(f"\n{'='*60}")
    print(f"INPUT: {instruction}")
    print(f"{'='*60}")
    output = run_classifier_agent(instruction)
    print(output)

## Step 10 — (Optional) Push Adapter to HuggingFace Hub

In [ ]:
# Only run this if you want to push to HF Hub
# from huggingface_hub import login
# login(token="hf_YOUR_TOKEN_HERE")

# model.push_to_hub("your-username/forgefactory-classifier-adapter")
# tokenizer.push_to_hub("your-username/forgefactory-classifier-adapter")
# print("✅ Pushed to HuggingFace Hub")

print("Uncomment above lines and add your HF token to push adapter to Hub")

---
## How to load this adapter later on the MI300X

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

BASE_MODEL   = "mistralai/Mistral-7B-v0.3"
ADAPTER_PATH = "./classifier_adapter_final"  # or HF Hub path

# On MI300X — load in fp16 (no quantization needed, you have 192GB)
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)
base      = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map="auto")
model     = PeftModel.from_pretrained(base, ADAPTER_PATH)
model.eval()

# Swap to searcher adapter later:
# model.load_adapter("./searcher_adapter")
```

---
## Notes
- Training 111 samples × 5 epochs on T4 ≈ **15–20 minutes**
- Adapter size ≈ **50–150MB** (not the full model)
- On MI300X: remove `BitsAndBytesConfig` and use `torch_dtype=torch.float16` — cleaner and faster
- To add more training data: just append more samples to the JSON and re-run